# Full PEC with fixed shot counts (one transpiled circuit, ibm_phoenix)

Prof. Goan (10 Oct): fix the number of shots and repeat with different shot counts to see the effect.

**Design.** The ISA circuit of the 10 Oct batch (`REF_BATCH`, seed `REF_SEED`) is loaded from QPY, not re-transpiled,
so the runs here can be compared directly with the three default-precision runs of that circuit. All other PEC settings
are unchanged: TREX + PEC, noise learned for all distinct layers, no overhead limit.

**What can be fixed under PEC (two check runs, 10 Oct).**
* Check 1, `default_shots = 51,200`: 363 randomizations x 1,600 shots = 580,800 shots, reported std 0.069.
* Check 2, `twirling.num_randomizations = 400`, `shots_per_randomization = 128`: 4,703 randomizations x 128 =
  601,984 shots, reported std 0.020.

PEC always multiplies the number of randomizations by the learned sampling overhead (4,703 ≈ 400 x 11.76), so the
total shot count cannot be fixed exactly; it is (requested randomizations) x (learned overhead) x 128, and the overhead
differs by a few percent from run to run because each run learns the noise anew. What can be fixed is the requested
number of randomizations and the shots per randomization. At almost the same total shots and QPU time (78 s each),
check 2 gave a 3.5x smaller std than check 1 (√(4,703/363) = 3.6): the PEC uncertainty is set by the number of
randomizations, not by shots per randomization.

**Levels.** 128 shots per randomization and `NUM_RAND_LIST` requested randomizations (≈ x12 after PEC scaling),
`N_REPEATS` independent runs each (own noise learning + own sampling). Check 2 is the first run of the lowest level.

**Second design (`DESIGN = "vary_spr"`).** The default 32 requested randomizations (≈ 350 after PEC scaling) are kept
fixed and the shots per randomization are varied (`SPR_LIST`: 128 / 512 / 2,048), `N_REPEATS` runs each, all submitted
together (`STAGE = "all"`). Expected: if the randomization-to-randomization variance dominates, the reported std hardly
changes; if shot noise dominates, it falls as 1/√(shots per randomization). Run on 10 Oct 23:14 with 32 randomizations
(128 / 512 / 2,048 shots).

**Matched total shots (`BASE_RAND = 400`, `SPR_LIST = [256, 512]`).** 400 × 256 and 400 × 512 have the same requested
total shots as 800 × 128 and 1,600 × 128, so each pair differs only in how the shots are allocated (more randomizations
vs more shots per randomization); 400 × 128 is the common starting point. Cell 8 prints these pairs side by side.

**Two stages (first design).**
1. `STAGE = "check"`: one run at the lowest level. Cell 7 prints the full metadata and checks that the requested
   randomizations and shots per randomization were applied (job metadata) and that shots = randomizations x 128.
2. `STAGE = "rest"`: the remaining runs, interleaved by level (800, 1600, 400, 800, 1600, 400, ...) in one Batch.
   Cell 5 refuses to continue unless the check run is saved and passed.

**What is tested.** If the run-to-run spread is statistical, both the reported std and the run-to-run spread
decrease roughly as 1/√(randomizations) (at 1,600 about half of the value at 400), and χ²/dof stays near 1 at
every level. If the spread does not decrease, it is systematic (e.g. noise learning or drift).

**Run order:** 1–3 → 4 (load + checks, no QPU) → 4b (circuit drawings, no QPU) → 5 (options, plan, budget; no QPU) → 6 (submit; returns
immediately) → 7 (collect; re-run until all DONE; safe after a kernel restart: run 1–3, then 7) → 8 (analysis).
Then set `STAGE = "rest"` and run 1–6 (incl. 4b) again, and 7–8 when done.

In [ ]:
# ── Cell 1: settings ──
from datetime import datetime, timezone
DRY_RUN       = False
SERVICE_NAME  = "NTU"
INSTANCE      = "<YOUR_INSTANCE_NAME>"
BACKEND_NAME  = "ibm_phoenix"
REF_BATCH     = "20261010_021000"   # 10 Oct batch whose ISA circuit is reused
REF_SEED      = 0                   # circuit 1 (seed 0); 2 for circuit 3
CZ_ERR_MAX    = 0.01                # only used to rebuild the 10 Oct file names
DESIGN        = "vary_spr"          # "fixed_rand": vary randomizations at 128 shots each (done 10 Oct 22:01)
                                    # "vary_spr":   32 randomizations, vary shots per randomization
SHOTS_PER_RAND = 128                # shots per randomization in "fixed_rand" (as in the default runs)
NUM_RAND_LIST = [400, 800, 1600]    # "fixed_rand": requested randomizations (PEC multiplies by the overhead, ~11)
BASE_RAND     = 400                 # "vary_spr": requested randomizations (32 = default, run 10 Oct 23:14)
SPR_LIST      = [256, 512]          # "vary_spr": shots per randomization; 400 x 128 already measured
                                    #   -> same requested totals as 800 x 128 and 1,600 x 128
LEVELS = ([(n, SHOTS_PER_RAND) for n in NUM_RAND_LIST] if DESIGN == "fixed_rand"
          else [(BASE_RAND, m) for m in SPR_LIST])            # (requested randomizations, shots per randomization)
N_REPEATS     = 3                   # independent runs per level
STAGE         = "all"               # "check" = 1 run at LEVELS[0]; "rest" = the others; "all" = every run at once
EXEC_MODE     = "batch"             # "batch" or "session"
GRP_ID, REP   = 0, 0
QPU_BUDGET_S  = 52 * 60             # QPU time left on this instance (11 Oct 00:30)
SEC_FIXED     = 63                  # planning: QPU s per run ~ 63 + 0.026 per 1,000 actual shots
SEC_PER_KSHOT = 0.026               #   (10 Oct: 64 s at ~48k shots; 78 s at ~0.6M shots)
OVERHEAD_EST  = 11.5                # learned overhead of this circuit (10.9-12.2 so far)
MAX_EXEC_S    = 900                 # cap per job (a capped job returns nothing and is still charged)
REF_RESULTS_CSV = f'pec_repeats_results_{BACKEND_NAME}.csv'   # 10 Oct default-precision runs (comparison)

In [ ]:
# ── Cell 2: imports + circuit (from Comparsion.ipynb) ──
"""
OTC-based Observable Comparison Experiment
比較 SV, Noise Model, FakeMarrakesh, ibm_marrakesh (Baseline/ZNE/Twirling/ZNE+Twirling/QESEM)
Group 1 (target=+0.8): Z0, Z4, Z6, X5, X7
Group 2 (target=-0.8): Z1, Z5, Z7, X4, X6
"""

import numpy as np
import pandas as pd
import csv, time
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.primitives import StatevectorEstimator
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel
from qiskit_ibm_runtime.fake_provider import FakeMarrakesh
from qiskit_ibm_runtime import QiskitRuntimeService, Session
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime.options import EstimatorOptions
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

import os, json, glob
from qiskit import qpy
from qiskit.quantum_info import PauliList
from qiskit_ibm_runtime import Batch, Session
from qiskit.transpiler import Target

def build_qru_circuit(data_input, hidden_input, params):
    """
    data_input:   numpy array, shape (8,)
    hidden_input: numpy array, shape (8,), 初始為全零
    params:       numpy array, shape (131,)
    """

    data   = QuantumRegister(4, 'data')
    hidden = QuantumRegister(4, 'hidden')
    anc    = QuantumRegister(2, 'anc')
    qc = QuantumCircuit(data, hidden, anc)

    # ==================
    # 1. ENCODING
    # ==================
    # Data: params[0:16], 每 qubit 4 個參數
    qc.barrier(label='ENC')
    for i in range(4):
        rx_angle = data_input[2*i] * params[4*i]     + params[4*i + 1]
        ry_angle = data_input[2*i + 1] * params[4*i + 2] + params[4*i + 3]
        qc.rx(rx_angle, data[i])
        qc.ry(ry_angle, data[i])

    # Hidden: params[16:32]
    for i in range(4):
        rx_angle = hidden_input[2*i] * params[16 + 4*i]     + params[16 + 4*i + 1]
        ry_angle = hidden_input[2*i + 1] * params[16 + 4*i + 2] + params[16 + 4*i + 3]
        qc.rx(rx_angle, hidden[i])
        qc.ry(ry_angle, hidden[i])

    # 循環 CX on data: 0→1→2→3→0
    for i in range(4):
        qc.cx(data[i], data[(i+1) % 4])

    # 循環 CX on hidden: 4→5→6→7→4
    for i in range(4):
        qc.cx(hidden[i], hidden[(i+1) % 4])

    # ==================
    # 2. UPDATE & RESET
    # ==================
    qc.barrier(label='UPD')
    qc.cx(data[0], hidden[0])
    qc.cswap(hidden[0], hidden[1], anc[0])

    # Rot on hidden[0]: params[32:35]
    qc.rz(params[32], hidden[0])
    qc.ry(params[33], hidden[0])
    qc.rz(params[34], hidden[0])

    qc.cswap(hidden[0], hidden[3], anc[1])
    # ancilla 保持 |0⟩，不需 reset

    # ==================
    # 3. VARIATIONAL LAYERS（4 層）
    # ==================
    idx = 35
    for layer in range(4):
        qc.barrier(label=f'VAR{layer+1}')
        # Rot on all non-ancilla qubits
        for reg in [data, hidden]:
            for i in range(4):
                qc.rz(params[idx],     reg[i]); idx += 1
                qc.ry(params[idx],     reg[i]); idx += 1
                qc.rz(params[idx],     reg[i]); idx += 1

        # 循環 CX on all non-ancilla: data[0..3] → hidden[0..3] → back
        all_q = [data[i] for i in range(4)] + [hidden[i] for i in range(4)]
        for i in range(8):
            qc.cx(all_q[i], all_q[(i+1) % 8])

    return qc

In [ ]:
# ── Cell 3: OTC parameters and observables (from Comparsion.ipynb) ──
# ── 設定 ──────────────────────────────────────────────────────
CALIB_CSV  = '../error_mitigation_comparison/qru_otc_multi_obs_20260503_205659.csv'
OUT_CSV = None
N_REPS     = 3      # 每組電路重複 3 次
N_SHOTS    = 4000

# ── Observable 分組 ───────────────────────────────────────────
# (qubit, pauli, obs_name, out_idx, group_id, target_val)
OBS_GROUPS = {
    0: {  # Group 1: target = +0.8
        'target': +0.8,
        'obs': [
            (0, 'Z', 'Z0',  0),
            (4, 'Z', 'Z4',  8),
            (6, 'Z', 'Z6', 12),
            (5, 'X', 'X5', 11),
            (7, 'X', 'X7', 15),
        ]
    },
    1: {  # Group 2: target = -0.8
        'target': -0.8,
        'obs': [
            (1, 'Z', 'Z1',  2),
            (5, 'Z', 'Z5', 10),
            (7, 'Z', 'Z7', 14),
            (4, 'X', 'X4',  9),
            (6, 'X', 'X6', 13),
        ]
    },
}

# ── OTC 資料庫讀取 ────────────────────────────────────────────
param_cols = [c for c in pd.read_csv(CALIB_CSV, nrows=0).columns if c.startswith('p')]
calib_df   = pd.read_csv(CALIB_CSV)

def get_params(group_id, target_val, rep_id):
    """從 OTC 資料庫取得電路參數"""
    obs_name = OBS_GROUPS[group_id]['obs'][0][2]  # 用第一個 obs 查詢
    mask = (
        (calib_df['group_id'] == group_id) &
        (calib_df['target_val'] == target_val) &
        (calib_df['rep_id'] == rep_id) &
        (calib_df['obs_name'] == obs_name)
    )
    rows = calib_df[mask]
    if len(rows) == 0:
        return None, None
    row     = rows.iloc[0]
    params  = row[param_cols].values.astype(float)
    sv_vals = {}
    for _, r in calib_df[(calib_df['group_id']==group_id) &
                          (calib_df['target_val']==target_val) &
                          (calib_df['rep_id']==rep_id)].iterrows():
        sv_vals[r['obs_name']] = r['sv_val']
    return params, sv_vals

# ── Observable 建構 ───────────────────────────────────────────
def make_obs_list(group_id, n=10):
    obs_list = []
    for q, p, name, _ in OBS_GROUPS[group_id]['obs']:
        s = ['I'] * n
        s[n-1-q] = p
        obs_list.append(SparsePauliOp(''.join(s)))
    return obs_list


RESULTS_CSV = f'pec_fixedshots_results_{BACKEND_NAME}.csv'
def status_name(job):
    s = job.status()
    return s if isinstance(s, str) else s.name
def cz_error_sum(target, circ):
    return float(sum(target[ins.operation.name][tuple(circ.find_bit(q).index for q in ins.qubits)].error
                     for ins in circ.data if ins.operation.num_qubits == 2 and ins.operation.name != 'barrier'))

In [ ]:
# ── Cell 4: backend, load the 10 Oct ISA circuit, verification (no QPU) ──
if DRY_RUN:
    backend = AerSimulator.from_backend(FakeMarrakesh()); calib = FakeMarrakesh().target
else:
    service = QiskitRuntimeService(name=SERVICE_NAME, instance=INSTANCE)
    backend = service.backend(BACKEND_NAME); calib = backend.target
target_val = OBS_GROUPS[GRP_ID]['target']
obs_names  = [o[2] for o in OBS_GROUPS[GRP_ID]['obs']]
params, sv_vals = get_params(GRP_ID, target_val, REP)
sv_arr = np.array([sv_vals[n] for n in obs_names])
logical_qc = build_qru_circuit(np.zeros(8), np.zeros(8), params)

def reduce_to_active(circ, ops):
    act = {circ.find_bit(q).index for ins in circ.data
           if ins.operation.name not in ('barrier', 'measure', 'delay', 'reset') for q in ins.qubits}
    for op in ops:
        act |= set(np.nonzero(np.any(op.paulis.z | op.paulis.x, axis=0))[0].tolist())
    act = sorted(act); m = {q: i for i, q in enumerate(act)}
    small = QuantumCircuit(len(act))
    for ins in circ.data:
        if ins.operation.name not in ('barrier', 'measure', 'delay', 'reset'):
            small.append(ins.operation, [m[circ.find_bit(q).index] for q in ins.qubits])
    return small, [SparsePauliOp(PauliList.from_symplectic(op.paulis.z[:, act], op.paulis.x[:, act]), op.coeffs)
                   for op in ops]

tag = f'{BACKEND_NAME}_seed{REF_SEED}_excl{int(CZ_ERR_MAX*100)}pct_{REF_BATCH}'
CIRCUIT_FILE  = f'pec_isa_circuit_{tag}.qpy'
OBS_FILE      = f'pec_observables_{tag}.json'
CIRCUITS_JSON = f'pec_repeats_circuits_{BACKEND_NAME}_{REF_BATCH}.json'

if DRY_RUN and not os.path.exists(CIRCUIT_FILE):          # dry run only: make local stand-in files
    pm = generate_preset_pass_manager(target=calib, optimization_level=1, seed_transpiler=REF_SEED)
    qc = pm.run(logical_qc); obs = [ob.apply_layout(qc.layout) for ob in make_obs_list(GRP_ID)]
    qpy.dump(qc, open(CIRCUIT_FILE, 'wb'))
    json.dump({n: op.to_list() for n, op in zip(obs_names, obs)}, open(OBS_FILE, 'w'), default=str)
    json.dump({'circuits': [{'seed': REF_SEED, 'n_cz': qc.count_ops().get('cz', 0), 'depth': qc.depth(),
                             'sum_cz_error': cz_error_sum(calib, qc)}]}, open(CIRCUITS_JSON, 'w'))

isa_qc  = qpy.load(open(CIRCUIT_FILE, 'rb'))[0]
obs_raw = json.load(open(OBS_FILE))
isa_obs = [SparsePauliOp.from_list([(p, complex(c)) for p, c in obs_raw[n]]) for n in obs_names]
ref_info = {c['seed']: c for c in json.load(open(CIRCUITS_JSON))['circuits']}[REF_SEED]

small, small_ops = reduce_to_active(isa_qc, isa_obs)
sv_check = np.max(np.abs(np.asarray(StatevectorEstimator().run([(small, small_ops)]).result()[0].data.evs) - sv_arr))
assert sv_check < 1e-6, f"loaded circuit does not reproduce the SV reference ({sv_check:.1e})"
n_cz = isa_qc.count_ops().get('cz', 0)
assert n_cz == ref_info['n_cz'] and isa_qc.depth() == ref_info['depth'], "circuit differs from the 10 Oct summary"
print(f"{backend.name} | {CIRCUIT_FILE} | SV check {sv_check:.1e} | depth {isa_qc.depth()}, CZ {n_cz}")
print(f"summed CZ error: {ref_info['sum_cz_error']:.3f} at 10 Oct transpile -> {cz_error_sum(calib, isa_qc):.3f} now")

In [ ]:
# ── Cell 4b: circuit drawings (no QPU) ──
# 'mpl' needs pylatexenc (pip install pylatexenc); falls back to text drawings if it is missing
from IPython.display import display
try:
    import pylatexenc; DRAW = 'mpl'
except ImportError:
    DRAW = 'text'; print("pylatexenc not installed -> text drawings")
names = [f'{r.name}{i}' for r in logical_qc.qregs for i in range(r.size)]
print("logical circuit (10 qubits): compare with the QRU circuit figure in the paper")
display(logical_qc.draw(DRAW, fold=40))
phys = ref_info.get('physical_qubits') or [int(q) for q in isa_qc.layout.final_index_layout()[:logical_qc.num_qubits]]
print(f"\n── ISA circuit {CIRCUIT_FILE}: depth {isa_qc.depth()}, "
      f"2q depth {isa_qc.depth(lambda ins: ins.operation.num_qubits == 2)}, CZ {n_cz}, "
      f"summed CZ error {ref_info['sum_cz_error']:.3f} (10 Oct) ──")
print("logical -> physical:", ", ".join(f"{n}->{p}" for n, p in zip(names, phys)))
print("gate counts:", dict(isa_qc.count_ops()))
display(isa_qc.draw(DRAW, idle_wires=False, fold=80, scale=0.5))

In [ ]:
# ── Cell 5: options, plan and budget check (no QPU) ──
def make_opts(n_rand, spr):
    o = EstimatorOptions()
    o.resilience_level = 1
    o.resilience.measure_mitigation = True
    o.resilience.pec_mitigation = not DRY_RUN
    o.resilience.layer_noise_learning.max_layers_to_learn = None
    o.resilience.pec.max_overhead = None
    if not DRY_RUN:                          # local simulator cannot run PEC; keep the dry run fast
        o.twirling.num_randomizations = n_rand       # PEC multiplies this by the learned overhead
        o.twirling.shots_per_randomization = spr
    if not DRY_RUN:
        o.max_execution_time = MAX_EXEC_S
    return o

CHECK_RUN = (0, LEVELS[0])                                       # (repeat, (n_rand, spr))
all_runs  = [(k, lv) for k in range(N_REPEATS) for lv in LEVELS]  # interleaved by level
if STAGE == "all":
    plan = all_runs
elif STAGE == "check":
    plan = [CHECK_RUN]
elif STAGE == "rest":
    plan = [r for r in all_runs if r != CHECK_RUN]
    if not DRY_RUN:                                                  # the check run must be saved and correct
        prev = pd.read_csv(RESULTS_CSV, dtype={'batch': str}) if os.path.exists(RESULTS_CSV) else pd.DataFrame()
        if len(prev) and 'design' not in prev: prev['design'] = 'default_shots'
        chk = prev[(prev['stage'] == 'check') & (prev['design'].fillna('default_shots') == DESIGN)] if len(prev) else prev
        ok = (len(chk) > 0 and bool((chk['shots'] == chk['num_randomizations'] * LEVELS[0][1]).all())
              and bool((chk['num_randomizations'] >= chk['num_randomizations_req']).all()))
        assert ok, "run STAGE='check' first and confirm shots_ok in Cell 7"
else:
    raise ValueError("STAGE must be 'check', 'rest' or 'all'")

used_s = 0.0
if os.path.exists(RESULTS_CSV):
    used_s = pd.read_csv(RESULTS_CSV)['qpu_s'].fillna(0).sum()
need_s = sum(SEC_FIXED + SEC_PER_KSHOT * n * m * OVERHEAD_EST / 1000 for _, (n, m) in plan)
print(f"design '{DESIGN}', stage '{STAGE}': {len(plan)} runs -> " + ", ".join(f"r{k}:{n}x{m}" for k, (n, m) in plan))
print(f"planning: ~{need_s/60:.0f} min QPU | used here so far {used_s/60:.0f} min | "
      f"budget {QPU_BUDGET_S/60:.0f} min -> {'OK' if used_s + need_s <= QPU_BUDGET_S else 'EXCEEDS'}")

In [ ]:
# ── Cell 6: SUBMIT (returns immediately) ──
label = time.strftime('%Y%m%d_%H%M%S')
jobs, jobs_local = [], {}
if DRY_RUN:
    for k, (n, m) in plan:
        j = Estimator(mode=backend, options=make_opts(n, m)).run([(isa_qc, isa_obs)]); jobs_local[j.job_id()] = j
        jobs.append({'shots_req': n * m, 'num_rand_req': n, 'spr_req': m, 'repeat': k, 'job_id': j.job_id()})
else:
    Mode = Batch if EXEC_MODE == "batch" else Session
    with Mode(backend=backend) as mode:
        ests = {lv: Estimator(mode=mode, options=make_opts(*lv)) for lv in LEVELS}   # options are per Estimator
        for k, (n, m) in plan:
            jobs.append({'shots_req': n * m, 'num_rand_req': n, 'spr_req': m, 'repeat': k,
                         'job_id': ests[(n, m)].run([(isa_qc, isa_obs)]).job_id()})
pending = {'label': label, 'stage': STAGE, 'design': DESIGN, 'shots_per_rand': SHOTS_PER_RAND, 'backend': BACKEND_NAME, 'instance': INSTANCE, 'exec_mode': EXEC_MODE,
           'submitted': time.strftime('%Y-%m-%d %H:%M:%S'), 'circuit_file': CIRCUIT_FILE,
           'ref_batch': REF_BATCH, 'ref_seed': REF_SEED, 'jobs': jobs}
with open(f'pec_fixedshots_pending_{BACKEND_NAME}_{label}.json', 'w') as f:       # local only (has job ids)
    json.dump(pending, f, indent=2)
with open('pec_job_ids.txt', 'a') as f:
    for j in jobs:
        f.write(f"{pending['submitted']} {BACKEND_NAME} fixedshots_{label} seed{REF_SEED} "
                f"shots{j['shots_req']} rep{j['repeat']} {j['job_id']}\n")
print(f"submitted {len(jobs)} job(s), stage '{STAGE}', in one {EXEC_MODE}; "
      f"saved pec_fixedshots_pending_{BACKEND_NAME}_{label}.json")

In [ ]:
# ── Cell 7: COLLECT — re-run until all jobs are DONE (safe after a kernel restart: run Cells 1-3 first) ──
def parse_ts(s):
    if s is None: return None
    return datetime.fromisoformat(str(s).replace('Z', '+00:00'))
obs_names = [o[2] for o in OBS_GROUPS[GRP_ID]['obs']]
_, sv_vals = get_params(GRP_ID, OBS_GROUPS[GRP_ID]['target'], REP)
sv_arr = np.array([sv_vals[n] for n in obs_names])
done = pd.read_csv(RESULTS_CSV, dtype={'batch': str}) if os.path.exists(RESULTS_CSV) else pd.DataFrame()
done_keys = set(zip(done['batch'], done['shots_req'], done['repeat'])) if len(done) else set()
if not DRY_RUN:
    svc = QiskitRuntimeService(name=SERVICE_NAME, instance=INSTANCE)
    bk = svc.backend(BACKEND_NAME)
new_rows, counts, circ_cache = [], {}, {}
for pf in sorted(glob.glob(f'pec_fixedshots_pending_{BACKEND_NAME}_*.json')):
    pend = json.load(open(pf))
    for jd in pend['jobs']:
        key = (str(pend['label']), jd['shots_req'], jd['repeat'])
        if key in done_keys:
            counts['saved'] = counts.get('saved', 0) + 1; continue
        if DRY_RUN and jd['job_id'] not in globals().get('jobs_local', {}):
            continue                                 # dry-run job from an earlier kernel
        job = jobs_local[jd['job_id']] if DRY_RUN else svc.job(jd['job_id'])
        if DRY_RUN: job.result()                     # local simulation: wait
        st = status_name(job); counts[st] = counts.get(st, 0) + 1
        if st != 'DONE':
            continue
        if pend['circuit_file'] not in circ_cache:
            circ_cache[pend['circuit_file']] = qpy.load(open(pend['circuit_file'], 'rb'))[0]
        circ = circ_cache[pend['circuit_file']]
        res = job.result(); meta = res[0].metadata
        if pend['stage'] == 'check':                 # full metadata of the check run, for manual inspection
            print(f"── check run {jd['job_id']} ──\njob metadata:", res.metadata, "\npub metadata:", meta)
        evs, stds = np.asarray(res[0].data.evs), np.asarray(res[0].data.stds)
        rel, z = (evs - sv_arr) / sv_arr, (evs - sv_arr) / stds
        ln_ = meta.get('resilience', {}).get('layer_noise', {})
        per_layer = ln_.get('unique_mitigated_layers_noise_overhead') or []
        shots_meta = meta.get('shots')
        t_run, cz_run, qpu = None, np.nan, np.nan
        if not DRY_RUN:
            qpu = job.usage()
            t_run = parse_ts((job.metrics().get('timestamps') or {}).get('running'))
            if t_run is not None:
                cz_run = cz_error_sum(bk.target_history(datetime=t_run), circ)
        design = pend.get('design', 'default_shots')
        spr_req = jd.get('spr_req', pend.get('shots_per_rand'))
        nrand_req = jd.get('num_rand_req', jd['shots_req'] // spr_req if design == 'fixed_rand' else None)
        nrand = meta.get('num_randomizations')
        jtw = (res.metadata or {}).get('twirling', {})      # options as applied (job metadata)
        applied = (nrand_req is not None and jtw.get('num_randomizations') == nrand_req
                   and jtw.get('shots_per_randomization') == spr_req)
        row = {'batch': str(pend['label']), 'stage': pend['stage'], 'design': design, 'shots_req': jd['shots_req'],
               'repeat': jd['repeat'], 'shots': shots_meta,
               'shots_ok': bool(applied and nrand is not None and nrand >= nrand_req
                                and shots_meta == nrand * spr_req),
               'num_randomizations_req': nrand_req, 'num_randomizations': nrand, 'shots_per_rand_req': spr_req,
               'started_utc': t_run.isoformat() if t_run else None, 'sum_cz_error_at_start': cz_run,
               'noise_overhead': ln_.get('noise_overhead'), 'mean_ln_layer_overhead':
               (float(np.mean(np.log(per_layer))) if per_layer else np.nan),
               'n_layers_total': ln_.get('total_mitigated_layers'), 'n_layers_unique': ln_.get('unique_mitigated_layers'),
               'qpu_s': qpu, 'mean_dev_pct': 100 * rel.mean(), 'min_dev_pct': 100 * rel.min(),
               'max_dev_pct': 100 * rel.max(), 'max_abs_z': float(np.max(np.abs(z))), 'mean_std': float(stds.mean())}
        row.update({f'ev_{n}': float(e) for n, e in zip(obs_names, evs)})
        row.update({f'std_{n}': float(s) for n, s in zip(obs_names, stds)})
        new_rows.append(row)
        if not row['shots_ok']:
            print(f"!! settings not applied as expected: requested {nrand_req} x {spr_req}, "
                  f"job metadata {jtw.get('num_randomizations')} x {jtw.get('shots_per_randomization')}, "
                  f"actual {nrand} randomizations / {shots_meta} shots -> check the metadata before STAGE='rest'")
        else:
            print(f"ok: requested {nrand_req} x {spr_req} -> {nrand} randomizations "
                  f"(x{nrand / nrand_req:.2f}, overhead {row['noise_overhead']:.2f}), {shots_meta:,} shots")
if new_rows:
    done = pd.concat([done, pd.DataFrame(new_rows)], ignore_index=True)
    done.to_csv(RESULTS_CSV, index=False)        # no job ids in this file
print("status:", counts, f"| {len(new_rows)} new results saved to {RESULTS_CSV} ({len(done)} total)")
if len(done):
    if 'design' not in done: done['design'] = 'default_shots'
    if 'shots_per_rand_req' not in done: done['shots_per_rand_req'] = np.nan
    print(done[['batch', 'stage', 'design', 'num_randomizations_req', 'shots_per_rand_req', 'repeat', 'shots',
                'num_randomizations', 'shots_ok',
                'noise_overhead', 'mean_dev_pct',
                'mean_std', 'max_abs_z', 'qpu_s']].sort_values(['design', 'num_randomizations_req', 'shots_per_rand_req', 'repeat'])
          .to_string(index=False, float_format=lambda v: f'{v:.3g}'))

In [ ]:
# ── Cell 8: analysis by shot count (10 Oct default-precision runs of the same circuit included) ──
import matplotlib.pyplot as plt
ev_cols, sd_cols = [f'ev_{n}' for n in obs_names], [f'std_{n}' for n in obs_names]
assert os.path.exists(RESULTS_CSV), "no results yet: run Cell 7 until jobs are DONE"
df = pd.read_csv(RESULTS_CSV, dtype={'batch': str}).copy()
if 'design' not in df: df['design'] = 'default_shots'
df['design'] = df['design'].fillna('default_shots')
if 'shots_per_rand_req' not in df: df['shots_per_rand_req'] = np.nan
df.loc[df['design'] == 'fixed_rand', 'shots_per_rand_req'] = df.loc[df['design'] == 'fixed_rand', 'shots_per_rand_req'].fillna(128)
def _label(d, r, n, m):
    if d in ('fixed_rand', 'vary_spr'): return f'requested {int(n):,} rand x {int(m):,} shots'
    return f'default_shots {int(r):,}'
df['level'] = [_label(d, r, n, m) for d, r, n, m in
               zip(df['design'], df['shots_req'], df['num_randomizations_req'], df['shots_per_rand_req'])]
parts = [df]
if os.path.exists(REF_RESULTS_CSV):
    ref = pd.read_csv(REF_RESULTS_CSV, dtype={'batch': str})
    ref = ref[(ref['batch'] == REF_BATCH) & (ref['seed'] == REF_SEED)].copy()
    ref['level'] = 'default (10 Oct)'; ref['shots_req'] = ref['shots']; ref['design'] = 'default'
    parts.insert(0, ref)
allr = pd.concat(parts, ignore_index=True)
if 'num_randomizations' not in allr: allr['num_randomizations'] = np.nan
allr['n_rand'] = pd.to_numeric(allr['num_randomizations'], errors='coerce')
allr['n_rand'] = allr['n_rand'].fillna(allr['shots'] / 128)   # 10 Oct default runs: 128 shots each
allr['spr'] = allr['shots'] / allr['n_rand']                    # actual shots per randomization
rank = {'default': 0, 'default_shots': 1, 'vary_spr': 2, 'fixed_rand': 2}
allr['n_req'] = pd.to_numeric(allr.get('num_randomizations_req'), errors='coerce')
allr['spr_req'] = pd.to_numeric(allr.get('shots_per_rand_req'), errors='coerce')
order = list(dict.fromkeys(allr.assign(_r=allr['design'].map(rank))
                           .sort_values(['_r', 'n_req', 'spr_req', 'shots'], na_position='first')['level']))

rows = []
for lvl in order:
    d = allr[allr['level'] == lvl]
    e, s = d[ev_cols].to_numpy(float), d[sd_cols].to_numpy(float)
    r = {'level': lvl, 'design': d['design'].iloc[0], 'n_req': d['n_req'].iloc[0], 'spr_req': d['spr_req'].iloc[0],
         'runs': len(d), 'shots': d['shots'].mean(),
         'n_rand': d['n_rand'].mean(), 'spr': d['spr'].mean(), 'qpu_s': pd.to_numeric(d['qpu_s'], errors='coerce').mean(),
         'overhead': d['noise_overhead'].mean(),
         'mean_dev_%': d['mean_dev_pct'].mean(),
         'runs_mean_dev_%': ', '.join(f'{v:+.1f}' for v in d.sort_values('repeat')['mean_dev_pct']),
         'reported_std': s.mean()}
    if len(d) >= 2:
        r['run_to_run_SD_dev_%'] = d['mean_dev_pct'].std(ddof=1)
        r['run_to_run_SD_obs'] = float(np.sqrt(e.var(axis=0, ddof=1).mean()))   # per observable, pooled
        w = 1 / s**2; mw = (w * e).sum(0) / w.sum(0)
        r['chi2/dof'] = float((((e - mw) / s)**2).sum() / (e.size - e.shape[1]))
    rows.append(r)
tab = pd.DataFrame(rows)
print(tab.drop(columns=['design', 'n_req', 'spr_req']).to_string(index=False, float_format=lambda v: f'{v:.3g}'))

fixed = tab[(tab['spr_req'] == 128) & (tab['n_req'] >= 400)]
if len(fixed) >= 2:
    b = fixed.iloc[0]
    print(f"\nrelative to {b['level']} (expected if statistical: x sqrt(N_ref/N), N = actual randomizations):")
    for _, r in fixed.iloc[1:].iterrows():
        exp_ = np.sqrt(b['n_rand'] / r['n_rand'])
        line = f"  {r['level']}: reported std x{r['reported_std']/b['reported_std']:.2f}"
        if 'run_to_run_SD_obs' in r and pd.notna(r.get('run_to_run_SD_obs')) and pd.notna(b.get('run_to_run_SD_obs')):
            line += f", run-to-run SD (per observable) x{r['run_to_run_SD_obs']/b['run_to_run_SD_obs']:.2f}"
        print(line + f"  (expected x{exp_:.2f})")

def _rr(r):
    v = r.get('run_to_run_SD_obs')
    return f"{v:.4f}" if pd.notna(v) else "  n/a "
for n_req, g in tab[tab['n_req'].notna()].groupby('n_req'):     # vary shots per randomization at fixed n_req
    g = g.sort_values('spr_req')
    if len(g) < 2: continue
    b = g.iloc[0]
    print(f"\nfixed {int(n_req)} requested randomizations, relative to {int(b['spr_req'])} shots each "
          f"(shot-noise limit x sqrt(s_ref/s); randomization variance dominated: x1.00):")
    for _, r in g.iloc[1:].iterrows():
        line = f"  x {int(r['spr_req']):,} shots: reported std x{r['reported_std']/b['reported_std']:.2f}"
        if pd.notna(r.get('run_to_run_SD_obs')) and pd.notna(b.get('run_to_run_SD_obs')):
            line += f", run-to-run SD (per observable) x{r['run_to_run_SD_obs']/b['run_to_run_SD_obs']:.2f}"
        print(line + f"  (shot-noise limit x{np.sqrt(b['spr'] / r['spr']):.2f})")

pairs = []                                                       # same requested total, different allocation
for _, a in tab[(tab['spr_req'] > 128) & (tab['n_req'] >= 400)].iterrows():
    m = tab[(tab['spr_req'] == 128) & (tab['n_req'] == a['n_req'] * a['spr_req'] / 128)]
    if len(m): pairs.append((m.iloc[0], a))
if pairs:
    print("\nsame requested total shots: more randomizations vs more shots per randomization")
    print(f"  {'requested total':>15} | {'allocation':>18} {'shots':>9} {'std':>7} {'run-to-run':>10} {'mean dev':>9} |"
          f" {'allocation':>18} {'shots':>9} {'std':>7} {'run-to-run':>10} {'mean dev':>9}")
    for r1, r2 in pairs:
        f = lambda r: (f"{int(r['n_req']):>6,} x {int(r['spr_req']):>5,} sh {r['shots']:>9,.0f} {r['reported_std']:>7.4f} "
                       f"{_rr(r):>10} {r['mean_dev_%']:>+8.1f}%")
        print(f"  {int(r1['n_req'] * 128):>15,} | {f(r1)} | {f(r2)}")

# plots
fig, ax = plt.subplots(1, 4, figsize=(20, 4.2))
for i, lvl in enumerate(order):
    d = allr[allr['level'] == lvl]
    ax[0].plot(d['n_rand'], d['mean_dev_pct'], 'o', color=f'C{i}', label=lvl)
ax[0].axhline(0, color='0.6', lw=0.8); ax[0].set_xscale('log')
ax[0].set_xlabel('randomizations (actual)'); ax[0].set_ylabel('mean deviation from SV (%)'); ax[0].legend(fontsize=8)
ax[1].loglog(tab['n_rand'], tab['reported_std'], 'o', label='mean reported std')
if 'run_to_run_SD_obs' in tab:
    ax[1].loglog(tab['n_rand'], tab['run_to_run_SD_obs'], 's', mfc='none', label='run-to-run SD (per observable)')
x = np.array([tab['n_rand'].min(), tab['n_rand'].max()])
ax[1].loglog(x, tab['reported_std'].iloc[0] * np.sqrt(tab['n_rand'].iloc[0] / x), ':', color='0.5', label='∝ 1/√randomizations')
ax[1].set_xlabel('randomizations (actual)'); ax[1].set_ylabel('std of expectation value'); ax[1].legend(fontsize=8)
xpos = np.arange(len(obs_names)); n_all = len(allr); width = 0.8 / max(n_all, 1); k = 0
for i, lvl in enumerate(order):
    for _, row in allr[allr['level'] == lvl].sort_values('repeat').iterrows():
        ax[2].errorbar(xpos - 0.4 + (k + 0.5) * width, row[ev_cols].to_numpy(float), yerr=row[sd_cols].to_numpy(float),
                       fmt='o', ms=3, color=f'C{i}', elinewidth=0.8); k += 1
ax[2].plot(xpos, sv_arr, '_', ms=30, color='k', label='SV'); ax[2].set_xticks(xpos, obs_names)
ax[2].set_ylabel('expectation value'); ax[2].legend(fontsize=8)
for i, (n_req, g) in enumerate(tab[tab['n_req'].notna()].groupby('n_req')):
    g = g.sort_values('spr_req')
    if len(g) < 2: continue
    ax[3].loglog(g['spr'], g['reported_std'], 'o-', color=f'C{i}', label=f'{int(n_req)} requested rand.: reported std')
    if 'run_to_run_SD_obs' in g:
        ax[3].loglog(g['spr'], g['run_to_run_SD_obs'], 's', mfc='none', color=f'C{i}', label=f'{int(n_req)}: run-to-run SD')
    xs = np.array([g['spr'].min(), g['spr'].max()])
    ax[3].loglog(xs, g['reported_std'].iloc[0] * np.sqrt(g['spr'].iloc[0] / xs), ':', color=f'C{i}')
ax[3].set_xlabel('shots per randomization (dotted: ∝ 1/√shots)'); ax[3].set_ylabel('std of expectation value')
ax[3].legend(fontsize=8)
fig.suptitle(f'Full PEC with fixed randomizations on {BACKEND_NAME} (10 Oct circuit, seed {REF_SEED})')
fig.tight_layout(); plt.show()